# 16. Reproducible hierarchy registries and scientific artifacts

![Reproducible evaluator pipeline](../images/16_reproducible_scientific_evaluators.svg)

**Learning goals:** validate the exact $2\times2\times8$ registry, verify nested manifests and stable digests, freeze seeds and random-stream versions, require final-step checkpoints, fail closed on resume or evaluation mismatch, and publish atomic summaries without participant identifiers.

In [ ]:
from collections import Counter
import hashlib
import json
import math
import os
from pathlib import Path
import tempfile
from typing import Literal

SEED = 16
SequenceSupport = Literal["low", "high"]
WindowPolicy = Literal["frozen_random", "resampled_anchor"]
SUPPORTS = ("low", "high")
POLICIES = ("frozen_random", "resampled_anchor")
print(f"registry example seed={SEED}")

## 1. Canonical content digests

A digest identifies exact canonical bytes. Sorting scientific identifiers and using fixed JSON separators prevents irrelevant dictionary field order from changing the digest. Lists remain order-sensitive, so a registry is sorted by its canonical cell key before hashing. A valid digest does not prove that low is nested within high, so nesting remains a separate invariant.

In [ ]:
def canonical_bytes(payload):
    return json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")

def digest(payload):
    return hashlib.sha256(canonical_bytes(payload)).hexdigest()

example_a = {"groups": ["g01", "g02"], "anchor_spacing": 8}
example_b = {"anchor_spacing": 8, "groups": ["g01", "g02"]}
assert digest(example_a) == digest(example_b)
print("canonical digest:", digest(example_a)[:16])

## 2. Construct an exact 32-row private registry

Each of eight blocks contains low-frozen, low-resampled, high-frozen, and high-resampled cells. Both policies at one support level reuse the same manifest. The low group set is a strict subset of the high group set. All four cells in a block share optimization and replicate seeds, and every row uses one frozen exposure.

In [ ]:
manifest_sequences = {}
manifest_anchors = {}
manifest_digests = {}
registry = []
training_exposure = 8_192_000
final_step = 128_000
frozen_fields = {
    "training_exposure": training_exposure, "anchor_spacing": 8,
    "window_seed_version": "window-v1",
    "temporal_stream_version": "temporal-v1",
    "spatial_stream_version": "spatial-v1",
    "mask_stream_version": "mask-v1",
    "protocol_revision": "hierarchy-v1",
}

def frozen_anchor(replicate, sequence_id):
    value = int(hashlib.sha256(f"{replicate}:{sequence_id}:window-v1".encode()).hexdigest()[:8], 16)
    return 8 * (value % 8)

for replicate in range(8):
    low_sequences = tuple(f"g{replicate:02d}_{index:03d}" for index in range(10))
    high_sequences = low_sequences + tuple(f"h{replicate:02d}_{index:03d}" for index in range(90))
    for support, sequences in (("low", low_sequences), ("high", high_sequences)):
        manifest = f"block_{replicate}/{support}.json"
        manifest_sequences[manifest] = set(sequences)
        manifest_anchors[manifest] = {sequence: frozen_anchor(replicate, sequence) for sequence in sequences}
        manifest_digests[manifest] = digest({"sequences": sorted(sequences),
                                             "frozen_anchors": manifest_anchors[manifest],
                                             "anchor_spacing": 8})
        for policy in POLICIES:
            model_label = f"b{replicate}_{support}_{policy}"
            registry.append({
                "model_label": model_label,
                "replicate": replicate,
                "sequence_support": support,
                "window_policy": policy,
                "train_manifest": manifest,
                "manifest_digest": manifest_digests[manifest],
                "unique_sequences": len(sequences),
                "optimization_seed": 10_000 + replicate,
                "replicate_seed": 20_000 + replicate,
                "final_checkpoint_identity": f"{model_label}:step-{final_step}",
                **frozen_fields,
            })
REGISTRY_FIELDS = {
    "model_label", "replicate", "sequence_support", "window_policy",
    "train_manifest", "manifest_digest", "unique_sequences",
    "optimization_seed", "replicate_seed", "final_checkpoint_identity",
    *frozen_fields,
}
assert len(registry) == 32
print(f"registry rows={len(registry)}; manifests={len(manifest_sequences)}")

## 3. Validate structure and cross-row invariants

Counting rows is not enough because a duplicate can hide a missing cell. Validation compares the observed key set with the full Cartesian product and rejects duplicate keys. It then checks seed sharing, manifest reuse, common exposure, sequence counts, digests, and strict low-within-high nesting.

In [ ]:
def validate_registry(rows, sequences_by_manifest, anchors_by_manifest, digests_by_manifest):
    if any(set(row) != REGISTRY_FIELDS for row in rows):
        raise ValueError("registry row schema changed")
    expected = {(r, s, w) for r in range(8) for s in SUPPORTS for w in POLICIES}
    keys = [(row["replicate"], row["sequence_support"], row["window_policy"]) for row in rows]
    if len(rows) != 32 or set(keys) != expected or any(count != 1 for count in Counter(keys).values()):
        raise ValueError("registry must contain exactly one row for every 2x2x8 cell")
    if any(row["model_label"] != f"b{row['replicate']}_{row['sequence_support']}_{row['window_policy']}"
           for row in rows):
        raise ValueError("model label does not match its canonical cell")
    if len({row["model_label"] for row in rows}) != 32:
        raise ValueError("model labels must be unique")
    if len({row["final_checkpoint_identity"] for row in rows}) != 32:
        raise ValueError("final checkpoint identities must be unique")
    if any(row["final_checkpoint_identity"] != f"{row['model_label']}:step-{final_step}" for row in rows):
        raise ValueError("final checkpoint identity mismatch")
    for field, expected_value in frozen_fields.items():
        if {row[field] for row in rows} != {expected_value}:
            raise ValueError(f"frozen field mismatch: {field}")
    for replicate in range(8):
        block = [row for row in rows if row["replicate"] == replicate]
        if {row["optimization_seed"] for row in block} != {10_000 + replicate}:
            raise ValueError("optimization seed does not match the frozen block seed")
        if {row["replicate_seed"] for row in block} != {20_000 + replicate}:
            raise ValueError("all four cells must share block seeds")
        by_support = {support: [row for row in block if row["sequence_support"] == support] for support in SUPPORTS}
        for support, pair in by_support.items():
            if len({row["train_manifest"] for row in pair}) != 1:
                raise ValueError("window-policy pair must reuse one manifest")
            manifest = pair[0]["train_manifest"]
            if {row["unique_sequences"] for row in pair} != {len(sequences_by_manifest[manifest])}:
                raise ValueError("actual sequence count does not match manifest")
            for row in pair:
                if row["manifest_digest"] != digests_by_manifest[row["train_manifest"]]:
                    raise ValueError("manifest digest mismatch")
        low_manifest = by_support["low"][0]["train_manifest"]
        high_manifest = by_support["high"][0]["train_manifest"]
        low_sequences = sequences_by_manifest[low_manifest]
        high_sequences = sequences_by_manifest[high_manifest]
        if not low_sequences < high_sequences:
            raise ValueError("low sequences must be a strict subset of high sequences")
        if any(anchors_by_manifest[low_manifest][sequence] != anchors_by_manifest[high_manifest][sequence]
               for sequence in low_sequences):
            raise ValueError("nested sequence changed frozen anchor")
    return True

def registry_rejected(candidate, anchors=manifest_anchors):
    try:
        validate_registry(candidate, manifest_sequences, anchors, manifest_digests)
    except ValueError:
        return True
    return False

assert validate_registry(registry, manifest_sequences, manifest_anchors, manifest_digests)
duplicate = [dict(row) for row in registry]
duplicate[-1] = dict(duplicate[-2])
stream_drift = [dict(row) for row in registry]
stream_drift[0]["mask_stream_version"] = "mask-v999"
wrong_count = [dict(row) for row in registry]
wrong_count[0]["unique_sequences"] += 1
wrong_anchors = {manifest: dict(values) for manifest, values in manifest_anchors.items()}
low_manifest = registry[0]["train_manifest"]
high_manifest = registry[2]["train_manifest"]
shared_sequence = sorted(manifest_sequences[low_manifest])[0]
wrong_anchors[high_manifest][shared_sequence] += 8
assert registry_rejected(duplicate)
assert registry_rejected(stream_drift)
assert registry_rejected(wrong_count)
assert registry_rejected(registry, wrong_anchors)
print("exact cell set, manifests, seeds, exposure, digests, and nesting validated")

## 4. Outcome-blind protocol freeze

The protocol snapshot fixes policies, exposure, margins, random-stream versions, final-step rule, and evaluator version before outcomes. A throughput rule may choose only between predeclared tiers. The snapshot digest reveals later changes, but it does not certify that the scientific choices are good.

In [ ]:
protocol = {
    "registry_digest": digest(sorted(registry, key=lambda row: (row["replicate"], row["sequence_support"], row["window_policy"]))),
    "sequence_supports": list(SUPPORTS),
    "window_policies": list(POLICIES),
    "training_exposure": training_exposure,
    "final_step": final_step,
    "window_seed_version": "window-v1",
    "stream_versions": {"temporal": "temporal-v1", "spatial": "spatial-v1", "mask": "mask-v1"},
    "gfc_version": "gfc-v2",
    "margins": {"simple": 0.0625, "interaction": 0.0625, "allocation": 0.0625, "gap": 0.0625},
    "exposure_selection_rule": "outcome-blind-throughput-v1",
}
protocol_digest = digest(protocol)
assert len(protocol_digest) == 64
print("frozen protocol digest:", protocol_digest[:16])

## 5. Final-step checkpoints and fail-closed validation

A filename such as `final.pt` is not provenance. Evaluation checks the checkpoint content digest, completed step, registry identity, manifest digest, seeds, policy, exposure, stream versions, and frozen protocol digest. Missing or different fields stop resume and evaluation.

In [ ]:
def checkpoint_record(row, content):
    return {
        "model_label": row["model_label"],
        "replicate": row["replicate"],
        "sequence_support": row["sequence_support"],
        "window_policy": row["window_policy"],
        "manifest_digest": row["manifest_digest"],
        "training_exposure": row["training_exposure"],
        "optimization_seed": row["optimization_seed"],
        "replicate_seed": row["replicate_seed"],
        "window_seed_version": row["window_seed_version"],
        "temporal_stream_version": row["temporal_stream_version"],
        "spatial_stream_version": row["spatial_stream_version"],
        "mask_stream_version": row["mask_stream_version"],
        "final_checkpoint_identity": row["final_checkpoint_identity"],
        "protocol_revision": row["protocol_revision"],
        "completed_step": protocol["final_step"],
        "protocol_digest": protocol_digest,
        "checkpoint_digest": hashlib.sha256(content).hexdigest(),
    }

def validate_checkpoint(record, row, content):
    expected = checkpoint_record(row, content)
    missing = set(expected) - set(record)
    unexpected = set(record) - set(expected)
    mismatched = {key for key, value in expected.items() if record.get(key) != value}
    if missing or unexpected or mismatched:
        problems = missing | unexpected | mismatched
        raise ValueError(f"checkpoint provenance mismatch: {sorted(problems)}")
    return True

row = registry[0]
checkpoint_bytes = b"synthetic-final-step-weights"
record = checkpoint_record(row, checkpoint_bytes)
assert validate_checkpoint(record, row, checkpoint_bytes)
tampered = dict(record, completed_step=127_999)
try:
    validate_checkpoint(tampered, row, checkpoint_bytes)
except ValueError:
    stopped_fail_closed = True
else:
    stopped_fail_closed = False
assert stopped_fail_closed
extra = dict(record, unregistered_stream_version="private-v1")
try:
    validate_checkpoint(extra, row, checkpoint_bytes)
except ValueError:
    extra_stopped_fail_closed = True
else:
    extra_stopped_fail_closed = False
assert extra_stopped_fail_closed
print("final-step checkpoint accepted; mismatched and extra provenance stopped")

## 6. Atomic, privacy-safe public summaries

Private tables may need participant identifiers for pairing, but public artifacts must not contain identifiers, participant rows, recording paths, embeddings, examples, or checkpoints. Build an allowlisted aggregate summary, validate it recursively, write a same-directory temporary file, parse it, and replace the destination only after every check passes.

In [ ]:
PUBLIC_KEYS = {"schema_version", "protocol_digest", "cell_means", "interaction", "interval"}
CELL_KEYS = {"LF", "LR", "HF", "HR"}
FORBIDDEN_KEY_PARTS = ("participant", "subject", "recording", "path", "embedding", "checkpoint")
FORBIDDEN_STRING_PARTS = ("/private/", "/users/", ".mp4", ".npy", ".npz", ".pt")

def finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)

def validate_public_payload(payload):
    if not isinstance(payload, dict) or set(payload) != PUBLIC_KEYS:
        raise ValueError("public payload does not match the allowlisted schema")
    def visit(value):
        if isinstance(value, dict):
            for key, nested in value.items():
                if not isinstance(key, str):
                    raise TypeError("public payload keys must be strings")
                if any(part in key.lower() for part in FORBIDDEN_KEY_PARTS):
                    raise ValueError("private key in public payload")
                visit(nested)
        elif isinstance(value, list):
            for nested in value:
                visit(nested)
        elif isinstance(value, str):
            lowered = value.lower()
            if any(part in lowered for part in FORBIDDEN_KEY_PARTS + FORBIDDEN_STRING_PARTS):
                raise ValueError("private value in public payload")
        elif not finite_number(value):
            raise TypeError("public payload contains an unsupported or non-finite value")
    visit(payload)
    if payload["schema_version"] != "hierarchy-summary-v1":
        raise ValueError("unknown public schema version")
    if (not isinstance(payload["protocol_digest"], str) or len(payload["protocol_digest"]) != 64
            or not set(payload["protocol_digest"]) <= set("0123456789abcdef")):
        raise ValueError("protocol digest must be a 64-character string")
    if not isinstance(payload["cell_means"], dict) or set(payload["cell_means"]) != CELL_KEYS or not all(
        finite_number(value) and 0.0 <= value <= 1.0 for value in payload["cell_means"].values()
    ):
        raise ValueError("cell means must contain four bounded numeric values")
    if not finite_number(payload["interaction"]):
        raise ValueError("interaction must be finite and numeric")
    interval = payload["interval"]
    if not isinstance(interval, list) or len(interval) != 2 or not all(finite_number(value) for value in interval):
        raise ValueError("interval must contain two finite numeric endpoints")
    if interval[0] > interval[1]:
        raise ValueError("interval endpoints are reversed")

def atomic_public_json(payload, destination, *, fail_before_replace=False):
    validate_public_payload(payload)
    destination = Path(destination)
    descriptor, temporary_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent)
    os.close(descriptor)
    temporary = Path(temporary_name)
    try:
        temporary.write_bytes(canonical_bytes(payload) + b"\n")
        if json.loads(temporary.read_text()) != payload:
            raise ValueError("read-back validation failed")
        if fail_before_replace:
            raise RuntimeError("injected failure")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

public_summary = {"schema_version": "hierarchy-summary-v1", "protocol_digest": protocol_digest,
                  "cell_means": {"LF": 0.42, "LR": 0.56, "HF": 0.58, "HR": 0.60},
                  "interaction": -0.12, "interval": [-0.17, -0.07]}
with tempfile.TemporaryDirectory() as directory:
    destination = Path(directory) / "summary.json"
    destination.write_text('{"generation": 1}\n')
    old_bytes = destination.read_bytes()
    try:
        atomic_public_json(public_summary, destination, fail_before_replace=True)
    except RuntimeError:
        pass
    assert destination.read_bytes() == old_bytes
    atomic_public_json(public_summary, destination)
    assert json.loads(destination.read_text()) == public_summary
path_injection = dict(public_summary)
path_injection["cell_means"] = dict(public_summary["cell_means"], LF="/private/healthgait/p001.mp4")
try:
    validate_public_payload(path_injection)
except ValueError:
    private_path_rejected = True
else:
    private_path_rejected = False
assert private_path_rejected
print("privacy-safe summary published atomically")

## Exercises and takeaways

1. Delete one registry cell and duplicate another. Why does a row-count check miss the error?
2. Break one low-within-high relationship while preserving both manifest digests. Which separate check fails?
3. Change only the mask-stream version in a checkpoint. Why must evaluation stop?
4. Add a nested `participant_id` field to the public summary. Confirm that privacy validation rejects it before publication.

**Takeaway:** reproducibility requires exact factorial structure, scientific nesting checks, content digests, complete seed and stream metadata, and a final-step rule. Fail-closed behavior protects lineage. Atomic allowlisted summaries protect both artifact integrity and participant privacy.

## Continue learning

[Previous notebook: 15](15_exposure_and_replication.ipynb) | [Lecture](../lectures/16_reproducible_scientific_evaluators.md) | [Curriculum](../README.md) | [Next notebook: 17](17_hierarchical_support_and_factorial_inference.ipynb)